In [ ]:
#uses the UN region mapping (https://github.com/lukes/ISO-3166-Countries-with-Regional-Codes) 
# to map aggregated data (6 csv files) into 1 by region instead of country
#edited: 6/25

In [3]:
import pandas as pd

#  Load export value and quantity files 
mining_val = pd.read_csv("../data/aggregated/mining_export_value.csv", index_col=0)
mining_qty = pd.read_csv("../data/aggregated/mining_export_quantity.csv", index_col=0)
manuf_val = pd.read_csv("../data/aggregated/manufacturing_export_value.csv", index_col=0)
manuf_qty = pd.read_csv("../data/aggregated/manufacturing_export_quantity.csv", index_col=0)
assembly_val = pd.read_csv("../data/aggregated/assembly_export_value.csv", index_col=0)
assembly_qty = pd.read_csv("../data/aggregated/assembly_export_quantity.csv", index_col=0)

#  Load country to region mapping 
region_map = pd.read_csv("../data/aggregated/all.csv")  # uploaded file
region_lookup = region_map.set_index("name")["region"].to_dict()

#  Function to map country to region 
def map_to_region(df):
    df = df.copy()
    df["region"] = df.index.map(region_lookup.get)
    return df.groupby("region").sum()

#  Compute total export value per region 
value_total = pd.DataFrame({
    "Mining": mining_val.sum(axis=1),
    "Manufacturing": manuf_val.sum(axis=1),
    "Assembly": assembly_val.sum(axis=1)
})
value_total_region = map_to_region(value_total)
value_total_region.to_csv("region_export_value.csv")

#  Compute export profitability (USD per ton) 
# Avoid div by zero
mining_usdpt = (mining_val / mining_qty.replace(0, float('nan'))).fillna(0)
manuf_usdpt = (manuf_val / manuf_qty.replace(0, float('nan'))).fillna(0)
assembly_usdpt = (assembly_val / assembly_qty.replace(0, float('nan'))).fillna(0)

# Compute average profitability per country (mean across all products)
profitability = pd.DataFrame({
    "Mining": mining_usdpt.mean(axis=1),
    "Manufacturing": manuf_usdpt.mean(axis=1),
    "Assembly": assembly_usdpt.mean(axis=1)
})
profitability_region = map_to_region(profitability)
profitability_region.to_csv("region_export_profitability.csv")
